# M1 Fault Gate Lock and Task/Activity Policy Redesign

        ## TL;DR
        - Fault는 19번에서 통과한 `fault_no_overlap + compact13 + RandomForest depth3` 조합을 좁은 threshold 구간에서 재검증한다.
        - Task/Activity는 모델을 더 키우는 대신 pre/post/short-window 후보의 coverage와 overlap, group CV 성능을 비교한다.
        - 모든 계산은 M1 원본 ZIP과 기존 M1 산출물만 사용한다.

## Context & Methods
        - 입력: `m1_4class_window_policy_audit.csv`, `m1_compact_feature_set_summary.csv`, 원본 `predist_dataset.zip`.
        - Fault lock: `compact13 + RandomForest(max_depth=3)`를 주 후보로 보고 threshold `0.45/0.50/0.55/0.60`을 확인한다.
        - Task/Activity redesign: 기존 7일 후보와 새 1일/3일 후보를 같은 feature schema로 재계산한다.
        - 검증: `substation_id` 기준 `StratifiedGroupKFold`.

        ### Key Assumptions
        - 짧은 1일/3일 window에서도 기존 154개 feature schema를 유지한다. 단, 7일 기준 temporal delta feature 중 이전 구간이 없는 값은 imputation된다.
        - `task_centered_event_window`는 원본에 event end/duration이 없어 audit 전용으로만 기록한다.
        - normal label은 회사 제공 35건을 그대로 유지한다.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import subprocess
import warnings
import zipfile

import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMClassifier = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "07_데이터산출물"
NOTEBOOK_DIR = PROJECT_ROOT / "06_노트북"
SOURCE_DIR = PROJECT_ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = SOURCE_DIR / "predist_dataset.zip"

WINDOW_AUDIT_PATH = OUTPUT_DIR / "m1_4class_window_policy_audit.csv"
FEATURE_SET_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"
OLD_19_DECISION_PATH = OUTPUT_DIR / "m1_event_type_one_vs_normal_decision_matrix.csv"
OLD_20_DECISION_PATH = OUTPUT_DIR / "m1_class_binary_boosting_decision_matrix.csv"

OUT_FAULT_THRESHOLD = OUTPUT_DIR / "m1_fault_gate_lock_threshold_metrics.csv"
OUT_FAULT_PRED = OUTPUT_DIR / "m1_fault_gate_lock_predictions.csv"
OUT_FAULT_DECISION = OUTPUT_DIR / "m1_fault_gate_lock_decision_matrix.csv"
OUT_POLICY_AUDIT = OUTPUT_DIR / "m1_task_activity_policy_audit.csv"
OUT_WINDOW_SUMMARY = OUTPUT_DIR / "m1_task_activity_window_candidate_summary.csv"
OUT_WINDOW_METRICS = OUTPUT_DIR / "m1_task_activity_window_candidate_metrics.csv"
OUT_WINDOW_PRED = OUTPUT_DIR / "m1_task_activity_window_candidate_predictions.csv"
OUT_REDESIGN_DECISION = OUTPUT_DIR / "m1_task_activity_redesign_decision_matrix.csv"
OUT_REPORT = OUTPUT_DIR / "21_M1_fault_gate_lock_task_activity_policy_redesign_보고서.md"

RANDOM_STATE = 42
N_SPLITS = 5
FAULT_THRESHOLDS = [0.45, 0.50, 0.55, 0.60]
POLICY_THRESHOLDS = [0.4, 0.5, 0.6]
ZIP_FOLDER = "manufacturer 1"

for p in [ZIP_PATH, WINDOW_AUDIT_PATH, FEATURE_SET_PATH, OLD_19_DECISION_PATH, OLD_20_DECISION_PATH]:
    assert p.exists(), p

print("project_root:", PROJECT_ROOT)
print("lightgbm_available:", LIGHTGBM_AVAILABLE)

project_root: C:\3rd_Project\HeatGridAgent
lightgbm_available: True


## Data Loading

In [2]:
window_audit = pd.read_csv(WINDOW_AUDIT_PATH)
feature_sets = pd.read_csv(FEATURE_SET_PATH)
old19 = pd.read_csv(OLD_19_DECISION_PATH)
old20 = pd.read_csv(OLD_20_DECISION_PATH)

def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes"])

def parse_feature_list(feature_set_name: str) -> list[str]:
    row = feature_sets.loc[feature_sets["feature_set"].eq(feature_set_name)]
    assert len(row) == 1, feature_set_name
    features = [f for f in str(row.iloc[0]["features"]).split("|") if f]
    missing = sorted(set(features) - set(window_audit.columns))
    assert not missing, f"{feature_set_name} missing in audit: {missing[:10]}"
    return features

FEATURE_COLUMNS = {
    "compact13": parse_feature_list("compact13_overlap"),
    "expanded154": parse_feature_list("expanded154"),
}
assert len(FEATURE_COLUMNS["compact13"]) == 13
assert len(FEATURE_COLUMNS["expanded154"]) == 154

META_COLUMNS = [
    "source_id", "row_kind", "final_class", "event_type", "source_event_id",
    "disturbance_row_id", "substation_id", "event_start", "window_policy",
    "window_start", "window_end", "window_days", "sample_count",
    "expected_sample_count", "coverage_rate", "coverage_eligible",
    "overlap_disturbance_count", "overlap_disturbance_types",
    "fault_event_id", "hard_normal_tag",
]

print("window_audit", window_audit.shape)
print({k: len(v) for k, v in FEATURE_COLUMNS.items()})

window_audit (298, 190)
{'compact13': 13, 'expanded154': 154}


## Feature Computation for New Task/Activity Windows

In [3]:
BASE_SIGNALS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
DERIVED_SIGNALS = [
    "s_hc1_supply_temperature_error",
    "p_net_delta_temperature",
    "p_net_power_flow_ratio",
    "p_return_gap",
]
ALL_SIGNALS = BASE_SIGNALS + DERIVED_SIGNALS
STAT_NAMES = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]

def read_zip_csv(path: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as z:
        with z.open(path) as f:
            return pd.read_csv(f, sep=";")

disturbances = read_zip_csv(f"{ZIP_FOLDER}/disturbances.csv")
normal_events = read_zip_csv(f"{ZIP_FOLDER}/normal_events.csv")
faults = read_zip_csv(f"{ZIP_FOLDER}/faults.csv")
disturbances = disturbances.rename(columns={"substation ID": "substation_id", "Event start": "event_start"})
disturbances["event_start"] = pd.to_datetime(disturbances["event_start"])
disturbances["disturbance_row_id"] = np.arange(1, len(disturbances) + 1)
disturbances["event_type"] = disturbances["type"].astype(str).str.lower()

normal_events = normal_events.rename(columns={"Event ID": "source_event_id", "substation ID": "substation_id"})
normal_events["Event start"] = pd.to_datetime(normal_events["Event start"])
normal_events["Event end"] = pd.to_datetime(normal_events["Event end"])

ops_cache: dict[int, pd.DataFrame] = {}

def load_ops(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in ops_cache:
        df = read_zip_csv(f"{ZIP_FOLDER}/operational_data/substation_{substation_id}.csv")
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)
        for col in BASE_SIGNALS:
            if col not in df.columns:
                df[col] = np.nan
        df["s_hc1_supply_temperature_error"] = df["s_hc1_supply_temperature"] - df["s_hc1_supply_temperature_setpoint"]
        df["p_net_delta_temperature"] = df["p_net_supply_temperature"] - df["p_net_return_temperature"]
        flow = df["p_net_meter_flow"].replace(0, np.nan)
        df["p_net_power_flow_ratio"] = df["p_net_meter_heat_power"] / flow
        df["p_return_gap"] = df["p_hc1_return_temperature"] - df["p_net_return_temperature"]
        ops_cache[substation_id] = df
    return ops_cache[substation_id]

def window_overlap_count(substation_id: int, start: pd.Timestamp, end: pd.Timestamp, exclude_row_id: int | None = None) -> tuple[int, str]:
    sub = disturbances.loc[disturbances["substation_id"].eq(int(substation_id))].copy()
    mask = sub["event_start"].ge(start) & sub["event_start"].lt(end)
    if exclude_row_id is not None:
        mask &= ~sub["disturbance_row_id"].eq(int(exclude_row_id))
    hits = sub.loc[mask]
    return int(len(hits)), "|".join(sorted(hits["event_type"].dropna().unique()))

def normal_collision_count(substation_id: int, start: pd.Timestamp, end: pd.Timestamp) -> int:
    sub = normal_events.loc[normal_events["substation_id"].eq(int(substation_id))]
    overlaps = sub["Event start"].lt(end) & sub["Event end"].gt(start)
    return int(overlaps.sum())

def expected_count(start: pd.Timestamp, end: pd.Timestamp) -> int:
    return int(round((end - start).total_seconds() / 600))

def stat_features(frame: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> dict[str, float]:
    expected = expected_count(start, end)
    out: dict[str, float] = {}
    window = frame.loc[frame["timestamp"].ge(start) & frame["timestamp"].lt(end)].copy()
    for signal in ALL_SIGNALS:
        values = pd.to_numeric(window[signal], errors="coerce") if signal in window else pd.Series(dtype=float)
        out[f"{signal}__mean"] = float(values.mean()) if len(values) else np.nan
        out[f"{signal}__std"] = float(values.std()) if len(values) else np.nan
        out[f"{signal}__min"] = float(values.min()) if len(values) else np.nan
        out[f"{signal}__max"] = float(values.max()) if len(values) else np.nan
        out[f"{signal}__median"] = float(values.median()) if len(values) else np.nan
        valid = values.dropna()
        out[f"{signal}__last_minus_first"] = float(valid.iloc[-1] - valid.iloc[0]) if len(valid) >= 2 else np.nan
        out[f"{signal}__missing_rate"] = float(values.isna().mean()) if len(values) else 1.0

    def mean_between(signal: str, left: pd.Timestamp, right: pd.Timestamp) -> float:
        seg = frame.loc[frame["timestamp"].ge(left) & frame["timestamp"].lt(right), signal]
        return float(pd.to_numeric(seg, errors="coerce").mean()) if len(seg) else np.nan

    def std_between(signal: str, left: pd.Timestamp, right: pd.Timestamp) -> float:
        seg = frame.loc[frame["timestamp"].ge(left) & frame["timestamp"].lt(right), signal]
        return float(pd.to_numeric(seg, errors="coerce").std()) if len(seg) else np.nan

    for signal in ALL_SIGNALS:
        last_1d_start = end - pd.Timedelta(days=1)
        last_12h_start = end - pd.Timedelta(hours=12)
        prev_12h_start = end - pd.Timedelta(hours=24)
        last_6h_start = end - pd.Timedelta(hours=6)
        prev_6h_start = end - pd.Timedelta(hours=12)
        out[f"{signal}__last_1d_mean_minus_prev_6d_mean"] = mean_between(signal, last_1d_start, end) - mean_between(signal, start, last_1d_start)
        out[f"{signal}__last_12h_mean_minus_prev_12h_mean"] = mean_between(signal, last_12h_start, end) - mean_between(signal, prev_12h_start, last_12h_start)
        out[f"{signal}__last_6h_mean_minus_prev_6h_mean"] = mean_between(signal, last_6h_start, end) - mean_between(signal, prev_6h_start, last_6h_start)
        out[f"{signal}__last_1d_std_minus_prev_6d_std"] = std_between(signal, last_1d_start, end) - std_between(signal, start, last_1d_start)

    out["sample_count"] = int(len(window))
    out["expected_sample_count"] = expected
    out["coverage_rate"] = float(len(window) / expected) if expected else 0.0
    return out

def compute_event_window_row(event: pd.Series, window_policy: str, days: int, direction: str) -> dict:
    event_start = pd.Timestamp(event["event_start"])
    if direction == "pre":
        start, end = event_start - pd.Timedelta(days=days), event_start
    elif direction == "post":
        start, end = event_start, event_start + pd.Timedelta(days=days)
    else:
        raise ValueError(direction)
    substation_id = int(event["substation_id"])
    ops = load_ops(substation_id)
    features = stat_features(ops, start, end)
    overlap_n, overlap_types = window_overlap_count(substation_id, start, end, int(event["disturbance_row_id"]))
    features.update({
        "source_id": f"disturbance_{int(event['disturbance_row_id']):04d}_{window_policy}",
        "row_kind": "event",
        "final_class": str(event["event_type"]),
        "event_type": str(event["event_type"]),
        "source_event_id": np.nan,
        "disturbance_row_id": int(event["disturbance_row_id"]),
        "substation_id": substation_id,
        "event_start": event_start,
        "window_policy": window_policy,
        "window_start": start,
        "window_end": end,
        "window_days": float(days),
        "coverage_eligible": bool(features["coverage_rate"] >= 0.95),
        "overlap_disturbance_count": overlap_n,
        "overlap_disturbance_types": overlap_types,
        "normal_collision_count": normal_collision_count(substation_id, start, end),
        "fault_event_id": np.nan,
        "hard_normal_tag": np.nan,
    })
    return features

print("disturbances by type")
print(disturbances["event_type"].value_counts().to_string())

disturbances by type
event_type
fault       67
activity    55
task        43


## Build Fault Lock and Policy Candidate Datasets

In [4]:
def normal_rows_from_audit() -> pd.DataFrame:
    normal = window_audit.loc[
        window_audit["final_class"].eq("normal")
        & window_audit["window_policy"].eq("normal_event_7d")
        & as_bool(window_audit["coverage_eligible"])
    ].copy()
    assert len(normal) == 35
    normal["normal_collision_count"] = 0
    return normal

def existing_event_rows(class_name: str, policy_variant: str) -> pd.DataFrame:
    eligible = as_bool(window_audit["coverage_eligible"])
    base = window_audit.loc[window_audit["final_class"].eq(class_name) & eligible].copy()
    if class_name == "fault":
        policies = ["fault_pre_7d"]; require_no_overlap = policy_variant == "no_overlap"
    elif class_name == "task":
        if policy_variant == "pre_all":
            policies = ["task_pre_7d"]; require_no_overlap = False
        elif policy_variant == "post_all":
            policies = ["task_post_7d"]; require_no_overlap = False
        elif policy_variant == "no_overlap":
            policies = ["task_post_7d", "task_pre_7d"]; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    elif class_name == "activity":
        if policy_variant == "pre_all":
            policies = ["activity_pre_7d"]; require_no_overlap = False
        elif policy_variant == "post_all":
            policies = ["activity_post_7d"]; require_no_overlap = False
        elif policy_variant == "no_overlap":
            policies = ["activity_pre_7d", "activity_post_7d"]; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    else:
        raise ValueError(class_name)
    if require_no_overlap:
        base = base.loc[pd.to_numeric(base["overlap_disturbance_count"], errors="coerce").fillna(0).eq(0)].copy()
    rank = {policy: i for i, policy in enumerate(policies)}
    base["policy_rank"] = base["window_policy"].map(rank)
    base = base.loc[base["policy_rank"].notna()].copy()
    base = base.sort_values(["disturbance_row_id", "policy_rank", "coverage_rate"], ascending=[True, True, False])
    base = base.drop_duplicates(subset=["disturbance_row_id"], keep="first").drop(columns=["policy_rank"])
    base["normal_collision_count"] = 0
    return base

def make_binary_dataset(target_class: str, dataset_name: str, event_rows: pd.DataFrame) -> pd.DataFrame:
    normal = normal_rows_from_audit()
    data = pd.concat([normal, event_rows], ignore_index=True).copy()
    data["dataset"] = dataset_name
    data["target_class"] = target_class
    data["binary_label"] = np.where(data["final_class"].eq(target_class), 1, 0)
    data["binary_label_name"] = np.where(data["binary_label"].eq(1), target_class, "normal")
    data["substation_id"] = data["substation_id"].astype(int)
    assert data["binary_label"].nunique() == 2, dataset_name
    return data

fault_no_overlap_rows = existing_event_rows("fault", "no_overlap")
fault_lock_dataset = make_binary_dataset("fault", "fault_no_overlap", fault_no_overlap_rows)

# Existing 7-day policy candidates.
policy_datasets: dict[str, pd.DataFrame] = {}
for target in ["task", "activity"]:
    for variant in ["pre_all", "post_all", "no_overlap"]:
        rows = existing_event_rows(target, variant)
        policy_datasets[f"{target}_{variant}"] = make_binary_dataset(target, f"{target}_{variant}", rows)

# New 1-day and 3-day task/activity candidates from the source ZIP.
new_policy_audit_rows = []
for target in ["task", "activity"]:
    target_events = disturbances.loc[disturbances["event_type"].eq(target)].copy()
    for direction in ["pre", "post"]:
        for days in [3, 1]:
            policy = f"{target}_{direction}_{days}d"
            rows = [compute_event_window_row(event, policy, days, direction) for _, event in target_events.iterrows()]
            cand = pd.DataFrame(rows)
            for col in FEATURE_COLUMNS["expanded154"]:
                if col not in cand.columns:
                    cand[col] = np.nan
            accepted = cand.loc[as_bool(cand["coverage_eligible"])].copy()
            policy_datasets[policy] = make_binary_dataset(target, policy, accepted)
            new_policy_audit_rows.append(cand)

new_policy_audit = pd.concat(new_policy_audit_rows, ignore_index=True)

# Audit-only centered rows: no event end/duration exists in raw task/activity disturbance records.
centered_audit = []
for target in ["task", "activity"]:
    target_events = disturbances.loc[disturbances["event_type"].eq(target)].copy()
    centered_audit.append(pd.DataFrame({
        "source_id": [f"disturbance_{int(x):04d}_centered_audit" for x in target_events["disturbance_row_id"]],
        "final_class": target,
        "event_type": target,
        "disturbance_row_id": target_events["disturbance_row_id"].astype(int).values,
        "substation_id": target_events["substation_id"].astype(int).values,
        "event_start": target_events["event_start"].values,
        "window_policy": f"{target}_centered_event_window",
        "audit_only": True,
        "audit_reason": "event_end_or_duration_not_available",
    }))
centered_audit = pd.concat(centered_audit, ignore_index=True)

print("fault_lock_dataset", fault_lock_dataset.shape)
print("policy datasets", {k: v.shape for k, v in policy_datasets.items()})

fault_lock_dataset (90, 195)
policy datasets {'task_pre_all': (72, 195), 'task_post_all': (76, 195), 'task_no_overlap': (72, 195), 'activity_pre_all': (82, 195), 'activity_post_all': (81, 195), 'activity_no_overlap': (81, 195), 'task_pre_3d': (75, 195), 'task_pre_1d': (75, 195), 'task_post_3d': (77, 195), 'task_post_1d': (77, 195), 'activity_pre_3d': (82, 195), 'activity_pre_1d': (82, 195), 'activity_post_3d': (81, 195), 'activity_post_1d': (81, 195)}


## Model Evaluation

In [5]:
def make_model(model_name: str, y_train: pd.Series | None = None):
    if model_name == "logistic_balanced":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)),
        ])
    if model_name == "random_forest_balanced_depth3":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == "extra_trees_balanced_depth3":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == "lightgbm_balanced":
        assert LIGHTGBM_AVAILABLE, "lightgbm unavailable"
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=120,
                learning_rate=0.03,
                max_depth=2,
                num_leaves=4,
                min_child_samples=5,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            )),
        ])
    raise ValueError(model_name)

def metric_row(y_true, y_pred, probability, threshold, metadata: dict) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    class_precision, class_recall, class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    try:
        roc = roc_auc_score(y_true, probability)
    except Exception:
        roc = np.nan
    return {
        **metadata,
        "threshold": threshold,
        "rows": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc,
        "normal_precision": class_precision[0],
        "normal_recall": class_recall[0],
        "normal_f1": class_f1[0],
        "target_precision": class_precision[1],
        "target_recall": class_recall[1],
        "target_f1": class_f1[1],
        "normal_support": int(support[0]),
        "target_support": int(support[1]),
        "normal_fpr": fp / (fp + tn) if (fp + tn) else 0,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def evaluate_dataset(data: pd.DataFrame, feature_set_name: str, model_names: list[str], thresholds: list[float]) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_cols = FEATURE_COLUMNS[feature_set_name]
    y = data["binary_label"].astype(int)
    groups = data["substation_id"].astype(int)
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    pred_rows = []
    fold_metric_rows = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(data, y, groups), start=1):
        train = data.iloc[train_idx].copy()
        test = data.iloc[test_idx].copy()
        train_groups = set(train["substation_id"].astype(int))
        test_groups = set(test["substation_id"].astype(int))
        group_overlap = len(train_groups & test_groups)
        assert group_overlap == 0
        X_train = train[feature_cols]
        y_train = train["binary_label"].astype(int)
        X_test = test[feature_cols]
        y_test = test["binary_label"].astype(int)
        for model_name in model_names:
            model = make_model(model_name, y_train)
            model.fit(X_train, y_train)
            probability = model.predict_proba(X_test)[:, 1]
            for _, row in test.iterrows():
                pred = {
                    "source_id": row["source_id"],
                    "final_class": row["final_class"],
                    "target_class": row["target_class"],
                    "source_event_id": row.get("source_event_id", np.nan),
                    "fault_event_id": row.get("fault_event_id", np.nan),
                    "disturbance_row_id": row.get("disturbance_row_id", np.nan),
                    "substation_id": int(row["substation_id"]),
                    "window_policy": row["window_policy"],
                    "window_start": row["window_start"],
                    "window_end": row["window_end"],
                    "coverage_rate": row["coverage_rate"],
                    "overlap_disturbance_count": row.get("overlap_disturbance_count", np.nan),
                    "normal_collision_count": row.get("normal_collision_count", np.nan),
                    "hard_normal_tag": row.get("hard_normal_tag", np.nan),
                    "dataset": row["dataset"],
                    "feature_set": feature_set_name,
                    "feature_count": len(feature_cols),
                    "model": model_name,
                    "fold": fold,
                    "y_true": int(row["binary_label"]),
                }
                pred_rows.append(pred)
            start_idx = len(pred_rows) - len(test)
            for j, prob in enumerate(probability):
                pred_rows[start_idx + j]["probability_target"] = float(prob)
                for threshold in thresholds:
                    suffix = str(threshold).replace(".", "_")
                    pred_rows[start_idx + j][f"prediction_t{suffix}"] = int(prob >= threshold)
            for threshold in thresholds:
                y_pred = (probability >= threshold).astype(int)
                fold_metric_rows.append(metric_row(y_test, y_pred, probability, threshold, {
                    "dataset": data["dataset"].iloc[0],
                    "target_class": data["target_class"].iloc[0],
                    "feature_set": feature_set_name,
                    "feature_count": len(feature_cols),
                    "model": model_name,
                    "fold": fold,
                    "metric_scope": "fold",
                    "train_rows": int(len(train)),
                    "test_rows": int(len(test)),
                    "train_groups": "|".join(map(str, sorted(train_groups))),
                    "test_groups": "|".join(map(str, sorted(test_groups))),
                    "group_overlap_count": group_overlap,
                }))
    predictions = pd.DataFrame(pred_rows)
    aggregate_metric_rows = []
    for model_name in model_names:
        subset = predictions.loc[predictions["model"].eq(model_name)].copy()
        y_true = subset["y_true"].astype(int)
        probability = subset["probability_target"].astype(float)
        for threshold in thresholds:
            suffix = str(threshold).replace(".", "_")
            y_pred = subset[f"prediction_t{suffix}"].astype(int)
            aggregate_metric_rows.append(metric_row(y_true, y_pred, probability, threshold, {
                "dataset": data["dataset"].iloc[0],
                "target_class": data["target_class"].iloc[0],
                "feature_set": feature_set_name,
                "feature_count": len(feature_cols),
                "model": model_name,
                "fold": "all",
                "metric_scope": "aggregate",
                "train_rows": np.nan,
                "test_rows": int(len(subset)),
                "train_groups": np.nan,
                "test_groups": np.nan,
                "group_overlap_count": 0,
            }))
    metrics = pd.DataFrame(fold_metric_rows + aggregate_metric_rows)
    return predictions, metrics

## Fault Gate Lock

In [6]:
fault_predictions_parts = []
fault_metrics_parts = []

pred, met = evaluate_dataset(
    fault_lock_dataset,
    "compact13",
    ["random_forest_balanced_depth3"],
    FAULT_THRESHOLDS,
)
fault_predictions_parts.append(pred)
fault_metrics_parts.append(met)

if LIGHTGBM_AVAILABLE:
    pred, met = evaluate_dataset(
        fault_lock_dataset,
        "expanded154",
        ["lightgbm_balanced"],
        FAULT_THRESHOLDS,
    )
    fault_predictions_parts.append(pred)
    fault_metrics_parts.append(met)

fault_predictions = pd.concat(fault_predictions_parts, ignore_index=True)
fault_metrics = pd.concat(fault_metrics_parts, ignore_index=True)

fault_metrics.to_csv(OUT_FAULT_THRESHOLD, index=False, encoding="utf-8-sig")
fault_predictions.to_csv(OUT_FAULT_PRED, index=False, encoding="utf-8-sig")

old19_fault = old19.loc[
    old19["target_class"].eq("fault")
    & old19["dataset"].eq("fault_no_overlap")
    & old19["feature_set"].eq("compact13")
    & old19["model"].eq("random_forest_balanced_depth3")
    & old19["threshold"].eq(0.5)
].iloc[0]
old20_fault = old20.loc[
    old20["target_class"].eq("fault")
    & old20["dataset"].eq("fault_no_overlap")
    & old20["feature_set"].eq("expanded154")
    & old20["model"].eq("lightgbm_balanced")
    & old20["threshold"].eq(0.6)
].iloc[0]

fault_agg = fault_metrics.loc[fault_metrics["metric_scope"].eq("aggregate")].copy()
fold_std = (
    fault_metrics.loc[fault_metrics["metric_scope"].eq("fold")]
    .groupby(["dataset", "feature_set", "model", "threshold"], as_index=False)
    .agg(
        fold_balanced_accuracy_std=("balanced_accuracy", "std"),
        fold_target_recall_std=("target_recall", "std"),
        fold_normal_fpr_std=("normal_fpr", "std"),
        max_group_overlap=("group_overlap_count", "max"),
    )
)
fault_decision = fault_agg.merge(fold_std, on=["dataset", "feature_set", "model", "threshold"], how="left")
fault_decision["passes_balanced_accuracy"] = fault_decision["balanced_accuracy"].ge(0.80)
fault_decision["passes_recall"] = fault_decision["target_recall"].ge(0.85)
fault_decision["passes_normal_fpr"] = fault_decision["normal_fpr"].le(0.25)
fault_decision["passes_group_overlap"] = fault_decision["max_group_overlap"].fillna(0).eq(0)
fault_decision["passes_lock_criteria"] = fault_decision[
    ["passes_balanced_accuracy", "passes_recall", "passes_normal_fpr", "passes_group_overlap"]
].all(axis=1)
main_mask = (
    fault_decision["feature_set"].eq("compact13")
    & fault_decision["model"].eq("random_forest_balanced_depth3")
)
main_sensitivity = fault_decision.loc[main_mask].copy()
main_pass_thresholds = main_sensitivity.loc[main_sensitivity["passes_lock_criteria"], "threshold"].tolist()
reproduced_19 = bool(
    np.isclose(
        fault_decision.loc[main_mask & fault_decision["threshold"].eq(0.5), "balanced_accuracy"].iloc[0],
        old19_fault["balanced_accuracy"],
    )
    and np.isclose(
        fault_decision.loc[main_mask & fault_decision["threshold"].eq(0.5), "target_recall"].iloc[0],
        old19_fault["recall_target"],
    )
    and np.isclose(
        fault_decision.loc[main_mask & fault_decision["threshold"].eq(0.5), "normal_fpr"].iloc[0],
        old19_fault["normal_fpr"],
    )
)
stable_threshold_count = len(main_pass_thresholds)
if not reproduced_19:
    fault_final_decision = "fault_gate_recheck_required"
elif 0.5 in main_pass_thresholds and stable_threshold_count >= 2:
    fault_final_decision = "fault_gate_locked"
elif 0.5 in main_pass_thresholds:
    fault_final_decision = "fault_gate_lock_pending_threshold_review"
else:
    fault_final_decision = "fault_gate_recheck_required"

fault_decision["old19_reference_balanced_accuracy"] = old19_fault["balanced_accuracy"]
fault_decision["old19_reference_recall"] = old19_fault["recall_target"]
fault_decision["old19_reference_normal_fpr"] = old19_fault["normal_fpr"]
fault_decision["old20_lightgbm_reference_balanced_accuracy"] = old20_fault["balanced_accuracy"]
fault_decision["old20_lightgbm_reference_recall"] = old20_fault["recall_target"]
fault_decision["old20_lightgbm_reference_normal_fpr"] = old20_fault["normal_fpr"]
fault_decision["reproduced_19_threshold_0_5"] = reproduced_19
fault_decision["stable_pass_threshold_count"] = stable_threshold_count
fault_decision["fault_final_decision"] = fault_final_decision
fault_decision.to_csv(OUT_FAULT_DECISION, index=False, encoding="utf-8-sig")

fault_decision[[
    "feature_set", "model", "threshold", "balanced_accuracy", "target_recall",
    "normal_fpr", "fp", "fn", "passes_lock_criteria", "fault_final_decision"
]]

,feature_set,model,threshold,balanced_accuracy,target_recall,normal_fpr,fp,fn,passes_lock_criteria,fault_final_decision
0,compact13,random_forest_balanced_depth3,0.45,0.835065,0.927273,0.257143,9,4,False,fault_gate_lock_pending_threshold_review
1,compact13,random_forest_balanced_depth3,0.50,0.845455,0.890909,0.200000,7,6,True,fault_gate_lock_pending_threshold_review
2,compact13,random_forest_balanced_depth3,0.55,0.790909,0.781818,0.200000,7,12,False,fault_gate_lock_pending_threshold_review
3,compact13,random_forest_balanced_depth3,0.60,0.764935,0.672727,0.142857,5,18,False,fault_gate_lock_pending_threshold_review
4,expanded154,lightgbm_balanced,0.45,0.806494,0.927273,0.314286,11,4,False,fault_gate_lock_pending_threshold_review
5,expanded154,lightgbm_balanced,0.50,0.807792,0.872727,0.257143,9,7,False,fault_gate_lock_pending_threshold_review
6,expanded154,lightgbm_balanced,0.55,0.803896,0.836364,0.228571,8,9,False,fault_gate_lock_pending_threshold_review
7,expanded154,lightgbm_balanced,0.60,0.832468,0.836364,0.171429,6,9,False,fault_gate_lock_pending_threshold_review


## Task/Activity Window Policy Evaluation

In [7]:
# Policy audit combines existing 7-day windows and newly computed short windows.
existing_policy_audit = []
for dataset_name, data in policy_datasets.items():
    event = data.loc[data["binary_label"].eq(1)].copy()
    existing_policy_audit.append(event)
policy_audit = pd.concat(existing_policy_audit + [centered_audit], ignore_index=True, sort=False)
policy_audit.to_csv(OUT_POLICY_AUDIT, index=False, encoding="utf-8-sig")

summary_rows = []
for dataset_name, data in policy_datasets.items():
    event = data.loc[data["binary_label"].eq(1)].copy()
    summary_rows.append({
        "dataset": dataset_name,
        "target_class": data["target_class"].iloc[0],
        "rows": len(data),
        "normal_rows": int((data["binary_label"] == 0).sum()),
        "target_rows": int((data["binary_label"] == 1).sum()),
        "substation_count": int(data["substation_id"].nunique()),
        "target_substation_count": int(event["substation_id"].nunique()),
        "target_overlap_rows": int(pd.to_numeric(event["overlap_disturbance_count"], errors="coerce").fillna(0).gt(0).sum()),
        "target_overlap_rate": float(pd.to_numeric(event["overlap_disturbance_count"], errors="coerce").fillna(0).gt(0).mean()) if len(event) else np.nan,
        "normal_collision_rows": int(pd.to_numeric(event.get("normal_collision_count", 0), errors="coerce").fillna(0).gt(0).sum()) if len(event) else 0,
        "coverage_min": float(pd.to_numeric(event["coverage_rate"], errors="coerce").min()) if len(event) else np.nan,
        "coverage_median": float(pd.to_numeric(event["coverage_rate"], errors="coerce").median()) if len(event) else np.nan,
        "window_policies": "|".join(sorted(event["window_policy"].dropna().astype(str).unique())),
    })
window_summary = pd.DataFrame(summary_rows)
window_summary.to_csv(OUT_WINDOW_SUMMARY, index=False, encoding="utf-8-sig")

policy_predictions_parts = []
policy_metrics_parts = []
model_names = ["logistic_balanced", "random_forest_balanced_depth3", "extra_trees_balanced_depth3"]
for dataset_name, data in policy_datasets.items():
    for feature_set_name in ["compact13", "expanded154"]:
        pred, met = evaluate_dataset(data, feature_set_name, model_names, POLICY_THRESHOLDS)
        policy_predictions_parts.append(pred)
        policy_metrics_parts.append(met)

policy_predictions = pd.concat(policy_predictions_parts, ignore_index=True)
policy_predictions.to_csv(OUT_WINDOW_PRED, index=False, encoding="utf-8-sig")
policy_metrics = pd.concat(policy_metrics_parts, ignore_index=True)
policy_metrics.to_csv(OUT_WINDOW_METRICS, index=False, encoding="utf-8-sig")

aggregate = policy_metrics.loc[policy_metrics["metric_scope"].eq("aggregate")].copy()
fold_std = (
    policy_metrics.loc[policy_metrics["metric_scope"].eq("fold")]
    .groupby(["dataset", "target_class", "feature_set", "model", "threshold"], as_index=False)
    .agg(
        fold_balanced_accuracy_std=("balanced_accuracy", "std"),
        fold_target_recall_std=("target_recall", "std"),
        fold_normal_fpr_std=("normal_fpr", "std"),
        max_group_overlap=("group_overlap_count", "max"),
    )
)
redesign = aggregate.merge(fold_std, on=["dataset", "target_class", "feature_set", "model", "threshold"], how="left")
redesign = redesign.merge(window_summary, on=["dataset", "target_class"], how="left", suffixes=("", "_summary"))
redesign["passes_overlap"] = redesign["target_overlap_rate"].le(0.25)
redesign["passes_balanced_accuracy"] = redesign["balanced_accuracy"].ge(0.75)
redesign["passes_recall"] = redesign["target_recall"].ge(0.75)
redesign["passes_normal_fpr"] = redesign["normal_fpr"].le(0.25)
redesign["passes_group_overlap"] = redesign["max_group_overlap"].fillna(0).eq(0)
redesign["candidate_pass"] = redesign[
    ["passes_overlap", "passes_balanced_accuracy", "passes_recall", "passes_normal_fpr", "passes_group_overlap"]
].all(axis=1)

class_decisions = {}
for target in ["task", "activity"]:
    sub = redesign.loc[redesign["target_class"].eq(target)].copy()
    passed = sub.loc[sub["candidate_pass"]].copy()
    if len(passed):
        passed["threshold_preference"] = (passed["threshold"] - 0.5).abs()
        passed = passed.sort_values(
            ["threshold_preference", "balanced_accuracy", "target_recall", "normal_fpr"],
            ascending=[True, False, False, True],
        )
    if len(passed):
        best = passed.iloc[0]
        if target == "activity" and str(best["dataset"]).startswith("activity_post"):
            decision = "activity_window_candidate_selected_post_state_detection"
        else:
            decision = f"{target}_window_candidate_selected"
        selected_dataset = best["dataset"]
    else:
        decision = f"{target}_label_review_required"
        selected_dataset = ""
    class_decisions[target] = (decision, selected_dataset)

redesign["target_policy_decision"] = redesign["target_class"].map(lambda x: class_decisions.get(x, ("", ""))[0])
redesign["selected_dataset_for_target"] = redesign["target_class"].map(lambda x: class_decisions.get(x, ("", ""))[1])

if fault_final_decision == "fault_gate_locked" and any("selected" in v[0] for v in class_decisions.values()):
    overall_decision = "partial_multiclass_ready"
elif fault_final_decision == "fault_gate_locked":
    overall_decision = "fault_gate_only_then_policy_redesign_continue"
else:
    overall_decision = "fault_gate_pending_and_policy_redesign_continue"
redesign["fault_final_decision"] = fault_final_decision
redesign["overall_decision"] = overall_decision
redesign.to_csv(OUT_REDESIGN_DECISION, index=False, encoding="utf-8-sig")

print("class_decisions", class_decisions)
print("overall_decision", overall_decision)
redesign.loc[redesign["threshold"].eq(0.5)].sort_values(["target_class", "balanced_accuracy"], ascending=[True, False]).groupby("target_class").head(5)[[
    "target_class", "dataset", "feature_set", "model", "threshold", "balanced_accuracy",
    "target_recall", "normal_fpr", "target_overlap_rate", "candidate_pass", "target_policy_decision"
]]

class_decisions {'task': ('task_window_candidate_selected', 'task_post_1d'), 'activity': ('activity_window_candidate_selected', 'activity_pre_1d')}
overall_decision fault_gate_pending_and_policy_redesign_continue


,target_class,dataset,feature_set,model,threshold,balanced_accuracy,target_recall,normal_fpr,target_overlap_rate,candidate_pass,target_policy_decision
202,activity,activity_pre_1d,compact13,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.042553,True,activity_window_candidate_selected
238,activity,activity_post_1d,compact13,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.086957,True,activity_window_candidate_selected
247,activity,activity_post_1d,expanded154,random_forest_balanced_depth3,0.5,0.989130,0.978261,0.000000,0.086957,True,activity_window_candidate_selected
205,activity,activity_pre_1d,compact13,extra_trees_balanced_depth3,0.5,0.985714,1.000000,0.028571,0.042553,True,activity_window_candidate_selected
211,activity,activity_pre_1d,expanded154,random_forest_balanced_depth3,0.5,0.978723,0.957447,0.000000,0.042553,True,activity_window_candidate_selected
130,task,task_pre_1d,compact13,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.400000,False,task_window_candidate_selected
139,task,task_pre_1d,expanded154,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.400000,False,task_window_candidate_selected
142,task,task_pre_1d,expanded154,extra_trees_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.400000,False,task_window_candidate_selected
166,task,task_post_1d,compact13,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.047619,True,task_window_candidate_selected
175,task,task_post_1d,expanded154,random_forest_balanced_depth3,0.5,1.000000,1.000000,0.000000,0.047619,True,task_window_candidate_selected


## Quality Checks and Report

In [8]:

# Independent QA and report generation from saved CSVs.
def markdown_table(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "(empty)"
    safe = frame.copy()
    for col in safe.columns:
        safe[col] = safe[col].map(lambda x: "" if pd.isna(x) else str(x).replace("|", "\\|"))
    header = "| " + " | ".join(map(str, safe.columns)) + " |"
    sep = "| " + " | ".join(["---"] * len(safe.columns)) + " |"
    rows = ["| " + " | ".join(map(str, row)) + " |" for row in safe.to_numpy()]
    return "\n".join([header, sep] + rows)


def recompute_from_predictions(predictions: pd.DataFrame, thresholds: list[float]) -> pd.DataFrame:
    keys = ["dataset", "target_class", "feature_set", "model"]
    rows = []
    for key, g in predictions.groupby(keys, dropna=False):
        y = g["y_true"].astype(int)
        for threshold in thresholds:
            suffix = str(threshold).replace(".", "_")
            col = f"prediction_t{suffix}"
            yp = g[col].astype(int)
            tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()
            rows.append(dict(zip(keys, key)) | {
                "threshold": threshold,
                "balanced_accuracy_recomputed": balanced_accuracy_score(y, yp),
                "macro_f1_recomputed": f1_score(y, yp, average="macro", zero_division=0),
                "tn_recomputed": int(tn),
                "fp_recomputed": int(fp),
                "fn_recomputed": int(fn),
                "tp_recomputed": int(tp),
            })
    return pd.DataFrame(rows)

fault_saved_metrics = pd.read_csv(OUT_FAULT_THRESHOLD)
fault_saved_predictions = pd.read_csv(OUT_FAULT_PRED)
fault_decision_saved = pd.read_csv(OUT_FAULT_DECISION)
policy_saved_metrics = pd.read_csv(OUT_WINDOW_METRICS)
policy_saved_predictions = pd.read_csv(OUT_WINDOW_PRED)
redesign_saved = pd.read_csv(OUT_REDESIGN_DECISION)
window_summary_saved = pd.read_csv(OUT_WINDOW_SUMMARY)

fault_recompute = recompute_from_predictions(fault_saved_predictions, FAULT_THRESHOLDS)
fault_agg = fault_saved_metrics.loc[fault_saved_metrics["metric_scope"].eq("aggregate")].copy()
fault_check = fault_agg.merge(fault_recompute, on=["dataset", "target_class", "feature_set", "model", "threshold"], how="left")
fault_ba_diff = float((fault_check["balanced_accuracy"] - fault_check["balanced_accuracy_recomputed"]).abs().max())
fault_f1_diff = float((fault_check["macro_f1"] - fault_check["macro_f1_recomputed"]).abs().max())

policy_recompute = recompute_from_predictions(policy_saved_predictions, POLICY_THRESHOLDS)
policy_agg = policy_saved_metrics.loc[policy_saved_metrics["metric_scope"].eq("aggregate")].copy()
policy_check = policy_agg.merge(policy_recompute, on=["dataset", "target_class", "feature_set", "model", "threshold"], how="left")
policy_ba_diff = float((policy_check["balanced_accuracy"] - policy_check["balanced_accuracy_recomputed"]).abs().max())
policy_f1_diff = float((policy_check["macro_f1"] - policy_check["macro_f1_recomputed"]).abs().max())

scan_files = [
    OUT_FAULT_THRESHOLD, OUT_FAULT_PRED, OUT_FAULT_DECISION, OUT_POLICY_AUDIT,
    OUT_WINDOW_SUMMARY, OUT_WINDOW_METRICS, OUT_WINDOW_PRED, OUT_REDESIGN_DECISION,
]
forbidden_terms = ["manufacturer" + "_2", "manufacturer" + " " + "2", "M" + "2"]
forbidden_hits = []
for path in scan_files:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    for term in forbidden_terms:
        if term in text:
            forbidden_hits.append((Path(path).name, term))

source_status = subprocess.run(
    ["git", "status", "--short", "--", str(SOURCE_DIR.relative_to(PROJECT_ROOT))],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
).stdout.strip()

special_needed = {20, 34, 69, 67, 19, 68, 35, 48}
special_ids = set(pd.to_numeric(pd.read_csv(OUTPUT_DIR / "m1_event_type_one_vs_normal_special_event_audit.csv")["event_id"], errors="coerce").dropna().astype(int))
special_retained = special_needed.issubset(special_ids)

quality = pd.DataFrame([
    {"check": "fault_metric_recompute", "pass": fault_ba_diff < 1e-12 and fault_f1_diff < 1e-12, "detail": f"ba_diff={fault_ba_diff}; f1_diff={fault_f1_diff}"},
    {"check": "task_activity_metric_recompute", "pass": policy_ba_diff < 1e-12 and policy_f1_diff < 1e-12, "detail": f"ba_diff={policy_ba_diff}; f1_diff={policy_f1_diff}"},
    {"check": "non_target_manufacturer_absent", "pass": len(forbidden_hits) == 0, "detail": str(forbidden_hits)},
    {"check": "source_zip_exists", "pass": ZIP_PATH.exists(), "detail": str(ZIP_PATH)},
    {"check": "source_metadata_clean", "pass": source_status == "", "detail": source_status},
    {"check": "special_event_policy_retained", "pass": special_retained, "detail": str(sorted(special_needed & special_ids))},
    {"check": "normal_35_retained", "pass": len(normal_rows_from_audit()) == 35, "detail": str(len(normal_rows_from_audit()))},
    {"check": "group_overlap_zero", "pass": fault_saved_metrics["group_overlap_count"].fillna(0).max() == 0 and policy_saved_metrics["group_overlap_count"].fillna(0).max() == 0, "detail": "max overlap 0"},
])
assert quality["pass"].all(), quality.to_string(index=False)

fault_lock_row = fault_decision_saved.loc[
    fault_decision_saved["feature_set"].eq("compact13")
    & fault_decision_saved["model"].eq("random_forest_balanced_depth3")
    & fault_decision_saved["threshold"].eq(0.5)
].iloc[0]
lightgbm_candidates = fault_decision_saved.loc[
    fault_decision_saved["feature_set"].eq("expanded154")
    & fault_decision_saved["model"].eq("lightgbm_balanced")
    & fault_decision_saved["threshold"].eq(0.6)
]
lightgbm_row = lightgbm_candidates.iloc[0] if len(lightgbm_candidates) else None
fault_final_decision_report = str(fault_lock_row["fault_final_decision"])
redesign_overall = str(redesign_saved["overall_decision"].dropna().iloc[0])
task_decision = str(redesign_saved.loc[redesign_saved["target_class"].eq("task"), "target_policy_decision"].dropna().iloc[0])
activity_decision = str(redesign_saved.loc[redesign_saved["target_class"].eq("activity"), "target_policy_decision"].dropna().iloc[0])

best_policy_t05 = (
    redesign_saved.loc[
        redesign_saved["threshold"].eq(0.5)
        & redesign_saved["candidate_pass"]
        & redesign_saved["dataset"].eq(redesign_saved["selected_dataset_for_target"])
    ]
    .sort_values(["target_class", "balanced_accuracy", "target_recall", "normal_fpr"], ascending=[True, False, False, True])
    .groupby("target_class")
    .head(1)
)

report_lines = [
    "# M1 Fault Gate Lock 및 Task/Activity Policy Redesign 보고서",
    "",
    "## 결론",
    f"- 최종 판단: `{redesign_overall}`",
    f"- fault 판단: `{fault_final_decision_report}`",
    f"- task 판단: `{task_decision}`",
    f"- activity 판단: `{activity_decision}`",
    "- fault는 19번 기준 조합을 재현했고, threshold 0.5에서 lock 기준을 만족했다.",
    "- 다만 fault는 threshold 0.5 한 지점만 통과해 잠금 보류 성격의 threshold review가 필요하다.",
    "- task/activity는 새 1일·3일 window까지 비교했고, overlap/recall/FPR 기준을 함께 본다.",
    "",
    "## 근거",
    "### Fault Lock",
    "- 기준 후보: `fault_no_overlap + compact13 + random_forest_balanced_depth3 + threshold 0.5`",
    f"- balanced accuracy `{fault_lock_row['balanced_accuracy']:.4f}`, recall `{fault_lock_row['target_recall']:.4f}`, normal FPR `{fault_lock_row['normal_fpr']:.4f}`, FP `{int(fault_lock_row['fp'])}`, FN `{int(fault_lock_row['fn'])}`",
    f"- 19번 결과 재현: `{bool(fault_lock_row['reproduced_19_threshold_0_5'])}`",
]
if lightgbm_row is not None:
    report_lines.append(f"- 비교 후보 `expanded154 + LightGBM + threshold 0.6`: BA `{lightgbm_row['balanced_accuracy']:.4f}`, recall `{lightgbm_row['target_recall']:.4f}`, normal FPR `{lightgbm_row['normal_fpr']:.4f}`")
report_lines.extend([
    "",
    markdown_table(fault_decision_saved[[
        "feature_set", "model", "threshold", "balanced_accuracy", "target_recall",
        "normal_fpr", "fp", "fn", "passes_lock_criteria", "fault_final_decision"
    ]]),
    "",
    "### Task/Activity Candidate Summary",
    markdown_table(window_summary_saved.sort_values(["target_class", "dataset"])),
    "",
    "### Task/Activity Selected Candidate Rows at Threshold 0.5",
    markdown_table(best_policy_t05[[
        "target_class", "dataset", "feature_set", "model", "threshold", "balanced_accuracy",
        "target_recall", "normal_fpr", "target_overlap_rate", "candidate_pass", "target_policy_decision"
    ]]),
    "",
    "## 한계",
    "- task/activity의 짧은 window는 기존 154개 temporal feature 중 일부가 비어 imputation에 의존한다.",
    "- task는 원본 event duration/end가 없어 centered window는 audit 전용으로만 남겼다.",
    "- fault lock은 class별 gate 기준이며 최종 4분류 전체 모델 확정은 아니다.",
    "- normal 35건 고정이라 FPR은 fold 구성에 민감하다.",
    "",
    "## 다음 작업 순서",
    "1. fault gate는 threshold 0.5 기준 잠금 후보로 유지하되, 한 지점 통과라 threshold review를 보고한다.",
    "2. task/activity는 선택 후보가 있으면 해당 window 기준으로 별도 재검증한다.",
    "3. 후보가 없으면 task/activity label 정의와 event duration 확보 가능성을 먼저 검토한다.",
    "4. fault + 하나 이상의 event type gate가 안정화되면 hierarchical 4분류 전략을 다시 설계한다.",
    "",
    "## 품질 검증",
    markdown_table(quality),
])
OUT_REPORT.write_text("\n".join(report_lines), encoding="utf-8")

print("fault_final_decision:", fault_final_decision_report)
print("task_decision:", task_decision)
print("activity_decision:", activity_decision)
print("overall_decision:", redesign_overall)
print(quality.to_string(index=False))


fault_final_decision: fault_gate_lock_pending_threshold_review
task_decision: task_window_candidate_selected
activity_decision: activity_window_candidate_selected
overall_decision: fault_gate_pending_and_policy_redesign_continue
                         check  pass                                                           detail
        fault_metric_recompute  True                                         ba_diff=0.0; f1_diff=0.0
task_activity_metric_recompute  True   ba_diff=1.1102230246251565e-16; f1_diff=1.1102230246251565e-16
non_target_manufacturer_absent  True                                                               []
             source_zip_exists  True C:\3rd_Project\HeatGridAgent\05_데이터셋\PreDist\predist_dataset.zip
         source_metadata_clean  True                                                                 
 special_event_policy_retained  True                                 [19, 20, 34, 35, 48, 67, 68, 69]
            normal_35_retained  True                     